# Teen Mental Health — AutoML + MLflow

Notebook de modelado: búsqueda automática del mejor algoritmo e hiperparámetros para clasificar `depression_label` usando **FLAML AutoML**, con seguimiento de experimentos en **MLflow**.

Este entorno corre sobre la imagen Docker definida en `Dockerfile` (dev container), que ya incluye todas las dependencias necesarias.

## 0. Dependencias

Todas las librerías ya están instaladas en la imagen del dev container (ver `Dockerfile`): no se requiere `pip install` en runtime.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
)

from pathlib import Path

from flaml import AutoML
import mlflow

RANDOM_STATE = 42

print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
print('mlflow :', mlflow.__version__)

---
## 1. Carga del dataset preprocesado

In [ ]:
df = pd.read_csv('../data/processed/teen_mental_health_preprocessed.csv')

TARGET = 'depression_label'
X = df.drop(columns=TARGET)
y = df[TARGET]

print(f'Shape: {df.shape}')
print(f'Features ({len(X.columns)}): {list(X.columns)}')
print('\nBalance del target:')
vc = y.value_counts()
print(pd.DataFrame({'n': vc, '%': (100 * vc / len(y)).round(2)}))

---
## 2. División train / test estratificada

Con una clase minoritaria muy pequeña (~2.6 % positivos), el split **estratificado** es obligatorio para que train y test mantengan la misma proporción de clases. Usamos 80/20.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'  Positivos: {y_train.sum()} ({100 * y_train.mean():.2f}%)')
print(f'Test:  {X_test.shape[0]} muestras')
print(f'  Positivos: {y_test.sum()} ({100 * y_test.mean():.2f}%)')

---
## 3. Tratamiento del desbalance: sample weights

Usamos **sample weights** inversamente proporcionales a la frecuencia de clase (`class_weight='balanced'`). Esto penaliza más los errores sobre la clase minoritaria (depresión) durante el entrenamiento, sin alterar el dataset (a diferencia de SMOTE).

In [ ]:
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

w0 = sample_weight[y_train == 0][0]
w1 = sample_weight[y_train == 1][0]
print(f'Peso clase 0 (no depresión): {w0:.4f}')
print(f'Peso clase 1 (depresión):    {w1:.4f}')
print(f'Ratio w1/w0: {w1 / w0:.1f}x  →  cada positivo pesa como {w1 / w0:.1f} negativos')

---
## 4. Configuración de MLflow

Los experimentos se registran localmente en `mlruns/` en la raíz del proyecto. La URI se construye con `Path.resolve().as_uri()` para obtener un `file:///ruta/absoluta` canónico, evitando ambigüedad según el directorio de trabajo desde el que se lance el kernel.

In [ ]:
MLFLOW_TRACKING_URI = 'sqlite:///' + str(Path('../mlflow.db').resolve())
EXPERIMENT_NAME = 'teen-mental-health-automl'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
exp = mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Experimento:   {exp.name}')
print(f'Tracking URI:  {MLFLOW_TRACKING_URI}')

---
## 5. AutoML con FLAML — objetivo: no dejar pasar ningún caso de depresión

FLAML realiza búsqueda bayesiana sobre el espacio conjunto de algoritmos e hiperparámetros, priorizando configuraciones prometedoras antes de explorar regiones costosas.

**Enfoque tipo screening (análogo a cáncer):** el costo de un falso negativo (no detectar depresión) es mucho mayor que el de un falso positivo (marcar un caso sano para revisión). Esto se traduce en dos decisiones separadas:

1. **Métrica de búsqueda — Average Precision (PR-AUC):** mide qué tan bien el modelo *ordena* los casos positivos por encima de los negativos en todo el rango de umbrales posibles. Un modelo trivial que prediga siempre positivo obtiene AP ≈ prevalencia (~2.6%, pésimo), así que la búsqueda sigue premiando modelos con capacidad real de discriminar — a diferencia de optimizar recall puro sobre una predicción binaria fija, que sí se maximiza trivialmente.
2. **Umbral de decisión (sección 7.3):** una vez elegido el mejor modelo, el umbral de clasificación se calibra aparte para maximizar recall en el holdout (idealmente 100%), reportando el costo en falsos positivos. Ahí es donde realmente se aplica el criterio "atrapar todos los casos, aunque sobren positivos".

**Algoritmos candidatos:**

| Alias | Algoritmo |
|---|---|
| `lgbm` | LightGBM |
| `xgboost` | XGBoost |
| `rf` | Random Forest |
| `extra_tree` | Extra Trees |
| `lrl1` | Logistic Regression (L1) |

In [ ]:
TIME_BUDGET = 180  # segundos
N_SPLITS    = 5

automl = AutoML()

automl_settings = dict(
    task='classification',
    metric='ap',  # average precision (PR-AUC): premia ordenar bien los positivos
    time_budget=TIME_BUDGET,
    eval_method='cv',
    n_splits=N_SPLITS,
    split_type='stratified',
    estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1'],
    sample_weight=sample_weight,
    seed=RANDOM_STATE,
    verbose=1,
)

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

print('Búsqueda AutoML completada.')

---
## 6. Resultados de la búsqueda AutoML

In [ ]:
print('=' * 55)
print(f'  Mejor estimador:      {automl.best_estimator}')
print(f'  Mejor AP (CV val):    {1 - automl.best_loss:.4f}')
print('=' * 55)
print('\nHiperparámetros óptimos:')
for k, v in automl.best_config.items():
    print(f'  {k}: {v}')

In [ ]:
print('Mejor AP de validación por estimador:')
rows = []
for est, loss in automl.best_loss_per_estimator.items():
    if loss is None:
        continue
    rows.append({'estimador': est, 'AP_cv': round(1 - loss, 4)})

per_est = pd.DataFrame(rows).set_index('estimador').sort_values('AP_cv', ascending=False)
per_est